In [1]:
import torch
from torch import nn
from datasets import load_dataset
from transformers import GPT2Tokenizer
import math
from torch.utils.data import DataLoader, dataset
from tqdm import tqdm
from tokenizers import ByteLevelBPETokenizer

from coreTransformer import *
from tokenizer import *
from trainingSetup import *
from dataloader import *
from inferenceTest import *

In [2]:
def initialize_weights(model):
    """More conservative weight initialization"""
    def _init_weights(module):
        if isinstance(module, nn.Linear):
            torch.nn.init.xavier_uniform_(module.weight, gain=0.01)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.xavier_uniform_(module.weight, gain=0.01)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.ones_(module.weight)
            torch.nn.init.zeros_(module.bias)
    model.apply(_init_weights)
    return model

In [3]:
def initialize_model(tokenizer, device='cuda'):
    model = SmallTransformer(
        vocab_size=tokenizer.get_vocab_size(),
        d_model=256,
        num_heads=8,
        num_layers=6,
        d_ff=1024,
        max_seq_len=512,
        dropout=0.1
    )
    
    # Initialize weights properly
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            module.weight.data.normal_(mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                module.bias.data.zero_()
        elif isinstance(module, nn.LayerNorm):
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)
    
    model.apply(_init_weights)
    return model.to(device)

In [4]:
# Initialize model and training components
num_epochs = 10
train_loader, val_loader, tokenizer = get_dataloaders(batch_size=8, tokenizer_type='byte-level-bpe')

model = SmallTransformer(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=512,  # Increased model capacity
    num_heads=8,
    num_layers=8,  # More layers
    d_ff=2048,    # Larger feed-forward network
    max_seq_len=512,
    dropout=0.1
)

# Train the model
train_model(model, train_loader, val_loader, num_epochs=num_epochs, patience=3)



Starting training on cuda
Number of batches: 2971
Accumulation steps: 16
Effective batch size: 128


Validating: 100%|██████████| 308/308 [00:09<00:00, 33.06it/s]



Epoch 1:
Average training loss: 5.1427
Average validation loss: 2.6046
Learning rate: 5.00e-05
Saving best model...


Validating: 100%|██████████| 308/308 [00:09<00:00, 33.18it/s]



Epoch 2:
Average training loss: 2.4698
Average validation loss: 2.4890
Learning rate: 1.00e-04
Saving best model...


...

Validating: 100%|██████████| 308/308 [00:09<00:00, 32.86it/s]



Epoch 9:
Average training loss: 2.2215
Average validation loss: 2.3499
Learning rate: 3.81e-06
Saving best model...


Validating: 100%|██████████| 308/308 [00:09<00:00, 33.14it/s]



Epoch 10:
Average training loss: 2.2185
Average validation loss: 2.3497
Learning rate: 0.00e+00
Saving best model...


(SmallTransformer(
   (embedding): Embedding(50000, 512, padding_idx=0)
   (layer_norm_embed): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
   (dropout): Dropout(p=0.1, inplace=False)
   (input_proj): Linear(in_features=512, out_features=512, bias=True)
   (transformer_blocks): ModuleList(
     (0): TransformerBlock(
       (norm1): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
       (norm2): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
       (attention): MultiHeadAttention(
         (q_proj): Linear(in_features=512, out_features=512, bias=True)
         (k_proj): Linear(in_features=512, out_features=512, bias=True)
         (v_proj): Linear(in_features=512, out_features=512, bias=True)
         (out_proj): Linear(in_features=512, out_features=512, bias=True)
         (dropout): Dropout(p=0.0125, inplace=False)
       )
       (ff): Sequential(
         (0): Linear(in_features=512, out_features=2048, bias=True)
         (1): GELU(approximate='none')
        

In [7]:

_, _, tokenizer = get_dataloaders(batch_size=8, tokenizer_type='byte-level-bpe')

model = SmallTransformer(
    vocab_size=tokenizer.get_vocab_size(),
    d_model=512,  # Increased model capacity
    num_heads=8,
    num_layers=8,  # More layers
    d_ff=2048,    # Larger feed-forward network
    max_seq_len=512,
    dropout=0.1
)
print("\nTesting inference:")
test_prompts = [
    "In the distant future,",
    "The scientist discovered",
    "Once upon a time in",
]

for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    try:
        generated = test_inference(
            model=model,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=50,
            temperature=0.3,  # Higher temperature for more randomness
            top_k=50
        )
        print(f"Generated: {prompt}{generated}")
    except Exception as e:
        print(f"Error generating text for prompt '{prompt}': {e}")


Testing inference:

Prompt: In the distant future,
Error loading model: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Generated: In the distant future,

Prompt: The scientist discovered
Error loading model: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Generated: The scientist discovered

Prompt: Once upon a time in
Error loading model: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider 